> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Local haplotype radix DAG — VIGOR 2.12.1

A reproducible phased fixture with founder paths, exact sequence and local gene
symbols. The default graph preserves observed paths and separate LD groups.
Ordinary 0/1/2 PLINK calls remain unphased genotype-pattern paths.

A 6 Mb `window_bp` means **total** width ±3 Mb. This compact fixture uses explicit
bounds. These tests establish graph correctness, not causal accuracy.

In [ ]:
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import networkx as nx
from vigor import (build_haplotype_radix_dag, haplotype_graph_variants,
    haplotype_graph_tables, haplotype_graph_features, write_haplotype_graph_zarr)
work = Path(tempfile.mkdtemp(prefix="vigor-graph-tutorial-"))
variants = pd.DataFrame({"chrom":"1", "pos":[3,7,11,15], "ref":"A", "alt":"G"},
                        index=["s1","s2","s3","s4"])
values=np.array([[[0,0,0,1],[1,0,0,1]],
                 [[0,1,0,1],[0,1,1,1]],
                 [[0,0,0,1],[0,1,1,1]]],dtype=np.uint8)
X=xr.DataArray(da.from_array(values,chunks=(2,2,2)),
    dims=("iid","haplotype","snp"),
    coords={"iid":["a","b","c"],"haplotype":[0,1],"snp":variants.index})
founders=pd.DataFrame([[0,0,0,2],[0,2,2,2]],index=["F1","F2"],columns=variants.index)
fasta=work/"reference.fa";fasta.write_text(">chr1\n"+"A"*30+"\n")
gtf=work/"genes.gtf";gtf.write_text(
 'chr1\tfixture\tgene\t2\t8\t.\t+\t.\tgene_id "g1"; gene_name "GeneA";\n'
 'chr1\tfixture\tgene\t11\t18\t.\t+\t.\tgene_id "g2"; gene_name "GeneB";\n')
G=build_haplotype_radix_dag(X,founders,variants=variants,
    reference_sequence_path=fasta,gtf_path=gtf,window_bounds=(1,20),
    hdbscan_min_cluster_size=2,hdbscan_min_samples=1)
assert nx.is_directed_acyclic_graph(G)
print(G.graph["path_kind"], G.number_of_nodes(), G.number_of_edges())

In [ ]:
paths=G.graph["paths"]
for row in paths.loc[~paths.is_founder].itertuples():
    animal=["a","b","c"].index(row.sample_id)
    alleles=values[animal,int(row.haplotype)]
    states=np.concatenate([G.nodes[node]["allele_states"] for node in row.nodes])
    assert np.array_equal(states,2*alleles)
    sequence="".join(G.nodes[node]["sequence"] for node in row.nodes)
    expected=list("A"*20)
    for pos,allele in zip(variants.pos,alleles):
        if allele:expected[pos-1]="G"
    assert sequence=="".join(expected)
vt=haplotype_graph_variants(G)
assert len(vt)==len(variants)
assert vt.loc["s1","genes"]==("GeneA",)
print(vt[["pos","genes","ld_component","hdbscan_cluster","ld_group"]])

Shared nodes can have multiple nesting labels. Edges describe observed
adjacency. Reconstruct paths from `G.graph["paths"]`; free traversal could
combine segments from different animals. LD edges are in `G.graph["ld_edges"]`
and do not indicate sequence transitions or historical recombination.

In [ ]:
for node,attrs in G.nodes(data=True):
    print(node,attrs["variants"],attrs["animals"],attrs["founders"],attrs["nesting_layers"])
features,animal_ids,feature_meta=haplotype_graph_features(G,min_samples=1)
assert features.shape[0]==3
assert features.max()<=2
print("Expert inputs:",features.shape,animal_ids.tolist())
node_tables=haplotype_graph_tables(G,format="xarray")
variant_dataset=haplotype_graph_variants(G,format="xarray")
assert variant_dataset.sizes["variant"]==4
store=write_haplotype_graph_zarr(G,work/"graph.zarr",chunk_rows=4,shard_rows=8)
restored=xr.open_zarr(store,group="variants",consolidated=True)
assert restored.sizes["variant"]==4
print("Zarr round trip passed")

The sparse feature adapter can supply genomic experts with node copy counts
and the variant table supplies SNP/LD context. This notebook does not fit a
fine-mapping posterior or demonstrate a prediction gain. Phenotype-dependent
peak selection and model fitting must be evaluated within training folds.

For a large window, record `ld_scope` and HDBSCAN status. Banded LD retains every
SNP but limits pairwise evidence; large incomplete LD components cannot be
assigned exact HDBSCAN clusters without additional LD computation.